https://www.kegg.jp/brite/query=00640&htext=br08901.keg&option=-a&node_proc=br08901_org&proc_enabled=rn&panel=collapse

Global and overview maps
01100 Metabolic pathways
01110 Biosynthesis of secondary metabolites
01120 Microbial metabolism in diverse environments
01200 Carbon metabolism
01210 2-Oxocarboxylic acid metabolism
01212 Fatty acid metabolism
01230 Biosynthesis of amino acids
01232 Nucleotide metabolism
01250 Biosynthesis of nucleotide sugars
01240 Biosynthesis of cofactors
01220 Degradation of aromatic compounds
01310 Nitrogen cycle
01320 Sulfur cycle

In [78]:
from bioservices.kegg import KEGG
from Bio.KEGG.KGML.KGML_parser import read
import pandas as pd
from tqdm import tqdm
from datetime import datetime

starttime = datetime.now()

# 初始化KEGG服务
k = KEGG()

# 读取数据
reactions_df = pd.read_csv(
    "../data/NRRL_1/7_Annotation/chr.kegg.reactions_with_genes_backup.csv",
    dtype={'Enzyme': str, 'Genes': str}
).fillna({'Enzyme': '', 'Genes': ''})

reactions_df = reactions_df.head(50)  # 选取前50行

reactions_df['PATHWAY'] = ""

avoid_pathways = {
    "rn01100", "rn01110", "rn01120", "rn01200", "rn01210", "rn01212",
    "rn01230", "rn01232", "rn01240", "rn01250", "rn01220", "rn01310", "rn01320"
}

# 遍历DataFrame的每一行（使用索引直接修改原数据）
for _, row in tqdm(reactions_df.iterrows(), total=len(reactions_df), desc="Processing reactions"):
    reaction_id = row['Reaction ID']
    
    # 获取KEGG数据并解析PATHWAY
    try:
        data = k.get(reaction_id)
        dict_data = k.parse(data)
        pathways = dict_data.get('PATHWAY', [])
        # 筛选目标通路
        filtered_pathways = [p for p in pathways if p not in avoid_pathways]
        row['PATHWAY'] = ';'.join(filtered_pathways)
        
        if not filtered_pathways:
            continue
            
        # 遍历每个目标通路，检查反应类型
        found = False
        for pathway_id in filtered_pathways:
            # 获取KGML数据
            kgml_data = k.get(pathway_id, "kgml")
            pathway_obj = read(kgml_data)  # 解析为Pathway对象
            
            # 遍历通路中的所有反应（转换为列表避免TypeError）
            for reaction in list(pathway_obj.reactions):
                # 检查当前反应是否匹配
                if reaction_id in reaction.name:  # 确保names包含反应ID
                    if reaction.type == "irreversible":
                        row['Reversible'] = "FALSE"
                    elif reaction.type == "reversible":
                        row['Reversible'] = "TRUE"
                    else:
                        row['Reversible'] = "UNKNOWN"
                    found = True
                    break  # 退出反应循环
            if found:
                break  # 退出通路循环
                
    except Exception as e:
        print(f"处理反应 {reaction_id} 时出错: {str(e)}")
        continue

# 保存修改后的DataFrame
reactions_df.to_csv("../data/NRRL_1/7_Annotation/chr.kegg.reactions_with_genes_backup_updated.csv", index=False)

endtime = datetime.now()
print(endtime - starttime)

WARNING [bioservices.KEGG:130]:  The URL (http://rest.kegg.jp) provided cannot be reached.
Processing reactions: 100%|██████████| 50/50 [03:40<00:00,  4.40s/it]

0:03:42.240677


Python中的dict_values对象

在Python中，dict_values是一个特殊的返回对象，用于表示字典中所有值的视图。这个视图是动态的，意味着如果字典发生变化，dict_values对象也会相应地变化。它不是一个列表，因此不支持索引操作，但可以通过list()函数转换为列表。

dict_values的使用

当你调用字典的values()方法时，你会得到一个dict_values对象。例如，如果你有一个字典dishes，并调用dishes.values()，你将得到一个包含所有字典值的dict_values对象。这个对象可以用于迭代，但不能直接进行索引访问。

dishes = {'eggs': 2, 'sausage': 1, 'bacon': 1, 'spam': 500}
values = dishes.values()

# 迭代dict_values对象
for val in values:
    print(val)

dict_values的特性

动态性：dict_values对象会随着字典的变化而变化。如果你从字典中删除一个项，dict_values对象也会相应地更新。
不支持索引：由于dict_values不是列表，你不能通过索引来访问元素。尝试这样做会引发TypeError。
转换为列表：虽然dict_values不支持索引，但你可以通过list()函数将其转换为列表，从而进行索引访问或其他列表操作。

示例
以下是一个使用dict_values的示例，展示了如何将其转换为列表，并演示了它的动态性。

# 创建一个字典
dishes = {'eggs': 2, 'sausage': 1, 'bacon': 1, 'spam': 500}

# 获取dict_values对象
values = dishes.values()

# 将dict_values转换为列表
values_list = list(values)
print(values_list) # 输出: [2, 1, 1, 500]

# 修改字典
dishes['eggs'] = 3
dishes['sausage'] = 2

# 再次转换dict_values为列表，观察变化
new_values_list = list(values)
print(new_values_list) # 输出: [3, 2, 1, 500]
在这个示例中，我们首先创建了一个字典dishes，然后获取了它的dict_value对象。我们将这个对象转换为列表，并打印出来。然后，我们修改了字典中的一些值，并再次将dict_values对象转换为列表。我们可以看到，新的列表反映了字典的变化。

结论
dict_values是Python字典中一个非常有用的特性，它提供了一种动态的方式来查看字典中的所有值。虽然它不支持索引，但可以很容易地转换为列表，以便进行更复杂的操作。理解dict_values的工作原理可以帮助你更有效地使用Python字典。

In [1]:
from bioservices.kegg import KEGG
from Bio.KEGG.KGML.KGML_parser import read
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import time

starttime = datetime.now()

# 初始化KEGG服务
k = KEGG()

# 读取数据（截取前50行）
reactions_df = pd.read_csv(
    "../data/NRRL_1/7_Annotation/chr.kegg.reactions_with_genes_backup.csv",
    dtype={'Enzyme': str, 'Genes': str}
).fillna({'Enzyme': '', 'Genes': ''})

# ----------- 显式初始化列 -----------
reactions_df['PATHWAY'] = ""  # 添加PATHWAY列

# 定义要排除的通路列表
avoid_pathways = {
    "rn01100", "rn01110", "rn01120", "rn01200", "rn01210", "rn01212",
    "rn01230", "rn01232", "rn01240", "rn01250", "rn01220", "rn01310", "rn01320"
}

# 遍历每一行（通过索引直接修改原DataFrame）
for idx in tqdm(reactions_df.index, total=len(reactions_df), desc="Processing reactions"):
    reaction_id = reactions_df.loc[idx, 'Reaction ID']
    
    try:
        # 获取KEGG数据并解析PATHWAY
        data = k.get(reaction_id)
        if not data:  # 跳过空响应
            continue
        dict_data = k.parse(data)
        pathways = dict_data.get('PATHWAY', [])
        
        # 筛选不在排除列表中的通路
        filtered_pathways = [p for p in pathways if p not in avoid_pathways]
        # 直接更新原DataFrame的PATHWAY列
        reactions_df.at[idx, 'PATHWAY'] = ';'.join(filtered_pathways)
        
        if not filtered_pathways:
            continue  # 无有效通路，跳过后续处理
            
        # 遍历每个通路，检查反应类型
        found = False
        for pathway_id in filtered_pathways:
            kgml_data = k.get(pathway_id, "kgml")
            if not kgml_data:  # 跳过空响应
                continue
            pathway_obj = read(kgml_data)
            
            # 遍历通路中的反应
            for reaction in list(pathway_obj.reactions):
                if reaction_id in reaction.name:
                    # 更新原DataFrame的Reversible列
                    reactions_df.at[idx, 'Reversible'] = True if reaction.type == "reversible" else False
                    found = True
                    break  # 退出反应循环
            if found:
                break  # 退出通路循环
                
    except Exception as e:
        print(f"处理反应 {reaction_id} 时出错: {str(e)}")
        continue

# 保存更新后的数据
reactions_df.to_csv("../data/NRRL_1/7_Annotation/chr.kegg.reactions_with_genes_backup_updated.csv", index=False)

endtime = datetime.now()
print(f"总耗时: {endtime - starttime}")

Processing reactions:  94%|█████████▍| 1532/1627 [55:17<01:14,  1.28it/s] WARNING [bioservices.KEGG:596]:  status is not ok with Not Found
WARNING [bioservices.KEGG:1181]:  Could not parse the entry correctly.
Processing reactions:  94%|█████████▍| 1536/1627 [55:20<01:03,  1.43it/s]WARNING [bioservices.KEGG:596]:  status is not ok with Bad Request
WARNING [bioservices.KEGG:1181]:  Could not parse the entry correctly.
Processing reactions:  94%|█████████▍| 1537/1627 [55:20<00:48,  1.84it/s]WARNING [bioservices.KEGG:596]:  status is not ok with Bad Request
WARNING [bioservices.KEGG:1181]:  Could not parse the entry correctly.
Processing reactions:  95%|█████████▍| 1538/1627 [55:20<00:48,  1.85it/s]WARNING [bioservices.KEGG:596]:  status is not ok with Bad Request
WARNING [bioservices.KEGG:1181]:  Could not parse the entry correctly.
Processing reactions:  95%|█████████▍| 1539/1627 [55:21<00:42,  2.07it/s]WARNING [bioservices.KEGG:596]:  status is not ok with Bad Request
WARNING [bioservi

总耗时: 0:56:01.061475


 FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'FALSE' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  reactions_df.at[idx, 'Reversible'] = "TRUE" if reaction.type == "reversible" else "FALSE"
  
这个警告是因为你在向 `reactions_df` 的 `Reversible` 列赋值时，值的类型与列原有的数据类型不兼容。

**错误原因**
• 原数据列类型：`Reversible` 列可能是 布尔型（`bool`） 或 数值型，而你试图将字符串（如 `"TRUE"`、`"FALSE"`）赋给该列。
• 类型冲突：字符串 `"TRUE"`/`"FALSE"` 与布尔值 `True`/`False` 或数值型不兼容。

**解决方案**
根据你的需求，选择以下两种方式之一：
**方法 1：统一使用布尔值**
如果 `Reversible` 列需要存储布尔值 (`True`/`False`)，直接赋值布尔类型：
```python
# 初始化时设为布尔型（如果原列不存在）
reactions_df['Reversible'] = False  # 默认值设为False

# 在循环中赋值时直接使用布尔值
reactions_df.at[idx, 'Reversible'] = (reaction.type == "reversible")
```

**方法 2：统一使用字符串**
如果希望保留字符串 `"TRUE"`/`"FALSE"`，需显式将列类型设为字符串：
```python
# 初始化时设为字符串类型（如果原列不存在）
reactions_df['Reversible'] = ""  # dtype会被推断为object（字符串）

# 在循环中赋值字符串
reactions_df.at[idx, 'Reversible'] = "TRUE" if reaction.type == "reversible" else "FALSE"
```

**注意事项**
1. 数据类型一致性：确保列初始化时的默认值与后续赋值类型一致。
2. 原始数据类型检查：如果 `Reversible` 列已存在，检查其类型：
   ```python
   print(reactions_df['Reversible'].dtype)  # 输出列类型
   ```
   如果类型为 `bool`，需先转换为字符串：
   ```python
   reactions_df['Reversible'] = reactions_df['Reversible'].astype(str)
   ```
通过显式控制数据类型，可以避免此警告并确保代码在未来的 Pandas 版本中兼容。

In [52]:
pathway = read(k.get("rn00410", "kgml"))
print(len(pathway.entries))
print(len(pathway.reactions))
print(len(pathway.maps))

84
44
8


In [31]:
pathway

In [32]:
pathway.reactions

dict_values([<Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B35FF770>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B5848550>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B5848690>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B35B2D70>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B35B2C40>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B5969C70>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B4740E20>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B4740F30>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B5446350>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B5446450>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B5471400>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B54714F0>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B58527B0>, <Bio.KEGG.KGML.KGML_pathway.Reaction object at 0x00000167B5852A50>, <Bio.KEGG.KGML.KGML_pathway.Reactio

In [33]:
pathway.reactions[0]

TypeError: 'dict_values' object is not subscriptable

In [53]:
type(pathway.reactions)

dict_values

In [38]:
# 将 reactions 的 values 转为列表
reactions_list = list(pathway.reactions)
# 访问第一个反应
first_reaction = reactions_list[0]

第一个反应的 ID: 41


In [43]:
print(first_reaction)

Reaction node ID: 41
Reaction KEGG IDs: rn:R03046
Type: reversible
Substrates: cpd:C02335
Products: cpd:C00894


In [47]:
first_reaction.type

'reversible'

In [57]:
first_reaction.name

'rn:R03046'

https://biopython.org/docs/latest/api/Bio.KEGG.KGML.KGML_pathway.html#Bio.KEGG.KGML.KGML_pathway.Reaction

KGML_pathway.py
KGML_parser.py

原因：for _, row in reactions_df.iterrows() 中的 row 是原始数据的副本，对其修改不会反映到原 DataFrame。
解决：改用 df.loc[index] 直接操作原 DataFrame。

在 Pandas 中截取 DataFrame 的前 N 行（例如前 50 行），有以下几种常用方法：

**方法 1：`head(n)`（推荐）**
直接使用 `head(n)` 获取前 `n` 行：
```python
df_first_50 = df.head(50)  # 选取前50行
```

**方法 2：`iloc` 切片**
通过位置索引 (`iloc`) 截取行范围：
```python
df_first_50 = df.iloc[0:50]  # 索引0到49（共50行）
```

**方法 3：直接切片**
对 DataFrame 直接切片（适用于默认整数索引）：
```python
df_first_50 = df[0:50]  # 效果等同 df.iloc[0:50]
```

**注意事项**
1. 索引非连续时：  
   如果 DataFrame 的索引不是连续的整数（例如自定义索引或删除过行），使用 `iloc` 更安全，因为它基于位置而非标签。

2. 行数不足时：  
   如果 DataFrame 总行数小于 `n`（例如只有30行），`head(50)` 会返回全部30行，不会报错。

3. 链式操作：  
   可结合其他操作（例如过滤后截取）：
   ```python
   # 先过滤列，再截取前50行
   df_filtered = df[['A', 'B']].head(50)
   ```

**扩展：截取中间行**
若需截取其他范围（如第50行到第100行）：
```python
df_middle = df.iloc[50:100]  # 索引50到99（共50行）
```

data\NRRL_1\7_Annotation\chr.kegg.reactions_with_genes_backup_updated.csv
-> 人工校正
data\NRRL_1\7_Annotation\chr.kegg.reactions_with_genes_backup_updated_twice.csv